# PHY368 notebook: Moseley's law, Einstein coefficients, radiative decay

Companion to the 13 October 2026 lecture. Run the cells top to bottom. The sliders use `ipywidgets`; if they do not display, the plain function calls underneath still work.

Every formula is the one used in the lecture: $E\approx10.2(Z-\sigma)^2$ eV, $B_{21}u/A_{21}=1/(e^x-1)$, $g_1B_{12}=g_2B_{21}$, $A_{21}/B_{21}=8\pi h\nu^3/c^3$, $\tau=1/\sum A$.

In [ ]:
import sys, os, csv
from pathlib import Path
BASE = Path.cwd()
if not (BASE / 'src').is_dir(): BASE = BASE.parent
sys.path.insert(0, str(BASE / 'src'))
import numpy as np, matplotlib.pyplot as plt
from phy368 import *
plt.rcParams.update({'font.size': 12})
rows = list(csv.DictReader(open(BASE / 'data/kalpha_tabulated.csv')))
el = [r['element'] for r in rows]; Zs = np.array([int(r['Z']) for r in rows]); Es = np.array([float(r['K_alpha1_eV']) for r in rows])
print('Loaded', len(rows), 'tabulated Kα1 lines (LBNL X-Ray Data Booklet, Table 1-2)')

## 1. Moseley explorer
Move σ and watch the model line against the tabulated Kα₁ points.

In [ ]:
def moseley_plot(sigma=1.0):
    z = np.linspace(15, 78, 200)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(z, np.sqrt(COEFF) * (z - sigma), color='#0F8B8D', lw=3, label=f'model, σ = {sigma:.2f}')
    ax[0].plot(Zs, np.sqrt(Es), 'o', color='#E07A1F', label='tabulated Kα₁')
    ax[0].set_xlabel('Z'); ax[0].set_ylabel('√E (eV$^{1/2}$)'); ax[0].legend()
    err = 100 * (kalpha_energy_eV(Zs, sigma) - Es) / Es
    ax[1].bar(el, err, color='#0F8B8D'); ax[1].axhline(0, color='k', lw=0.8); ax[1].set_ylabel('model − tabulated (%)')
    plt.tight_layout(); plt.show()
try:
    from ipywidgets import interact, FloatSlider
    interact(moseley_plot, sigma=FloatSlider(min=0, max=2, step=0.05, value=1.0))
except Exception:
    moseley_plot(1.0)

In [ ]:
# Worked example: copper
E = kalpha_energy_eV(29)
print(f'Cu: E = {E:.1f} eV = {E/1000:.2f} keV, λ = {wavelength_nm(E):.4f} nm, tabulated Kα1 = 8047.78 eV')
for name, Z, E_t in zip(el, Zs, Es):
    print(f'{name:>2} Z={Z:2d}  effective σ = {effective_sigma(Z, E_t):6.3f}')

## 2. Stimulated against spontaneous emission
$B_{21}u/A_{21}=1/(e^x-1)$, $x=hc/(\lambda k_BT)$.

In [ ]:
def ratio_plot(T=300.0):
    lam = np.logspace(-7, -1, 400)
    r = [stim_over_spont(l, T) for l in lam]
    fig, ax = plt.subplots(figsize=(7, 4)); ax.loglog(lam * 1e9, r, color='#0F8B8D', lw=3)
    ax.axhline(1, ls='--', color='grey'); ax.set_ylim(1e-50, 1e6)
    ax.set_xlabel('wavelength (nm)'); ax.set_ylabel('B21 u / A21'); ax.set_title(f'T = {T:.0f} K'); plt.show()
try:
    interact(ratio_plot, T=FloatSlider(min=100, max=10000, step=100, value=300))
except Exception:
    ratio_plot(300)
print('500 nm, 300 K :', stim_over_spont(500e-9, 300))
print('3 cm,   300 K :', stim_over_spont(0.03, 300))

## 3. Two-level saturation
With $r=B_{21}u/A_{21}$, steady state gives $N_2/N_1=(g_2/g_1)\,r/(1+r)$. The ratio $(N_2/g_2)/(N_1/g_1)=r/(1+r)<1$ always, so a two-level system cannot be inverted by radiation alone.

In [ ]:
r = np.logspace(-3, 3, 200)
plt.semilogx(r, r / (1 + r), color='#E07A1F', lw=3); plt.axhline(1, ls='--', color='grey')
plt.xlabel('r = B21 u / A21'); plt.ylabel('(N2/g2) / (N1/g1)'); plt.ylim(0, 1.1); plt.show()

## 4. Radiative decay: Monte Carlo ensemble
Each atom decays at a random time drawn from an exponential with mean τ = 1/ΣA, and picks a channel with probability A/ΣA.

In [ ]:
def decay_demo(A_a=4.0, A_b=1.0, k_nr=0.0, N=400, seed=368):
    rates = np.array([A_a, A_b, k_nr]) * 1e7
    tot = rates.sum()
    if tot == 0:
        print("No decay: lifetime is infinite, surviving fraction is 1, branching is undefined.")
        return
    tau = 1 / tot
    rng = np.random.default_rng(seed)
    t = rng.exponential(tau, N); ch = rng.choice(3, size=N, p=rates / tot)
    tt = np.linspace(0, 5 * tau, 300)
    frac = [(t > s).mean() for s in tt]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(tt * 1e9, frac, color='#E07A1F', lw=3, label='simulated'); ax.plot(tt * 1e9, np.exp(-tt / tau), '--', color='#14213D', label='theory')
    ax.set_xlabel('t (ns)'); ax.set_ylabel('fraction excited'); ax.legend(); plt.show()
    print(f'tau = {tau*1e9:.2f} ns   expected branching = {np.round(rates/tot,3)}   simulated = {np.round(np.bincount(ch, minlength=3)/N,3)}')
try:
    from ipywidgets import IntSlider
    interact(decay_demo, A_a=FloatSlider(min=0, max=10, step=0.5, value=4), A_b=FloatSlider(min=0, max=10, step=0.5, value=1), k_nr=FloatSlider(min=0, max=10, step=0.5, value=0), N=IntSlider(min=100, max=5000, step=100, value=400), seed=IntSlider(min=1, max=999, value=368))
except Exception:
    decay_demo()

## 5. Check yourself
Edit the numbers and re-run. These are the worksheet and practice problems.

In [ ]:
tau, b = lifetime_and_branching([6e7, 3e7, 1e7])
print(f'three channels: tau = {tau*1e9:.1f} ns, branching = {b}, left after 25 ns = {surviving_fraction(25e-9, tau):.4f}')
print('Synthetic Kα line 17.15 keV -> Z =', round(z_from_energy(17150), 2))
print('g1=3, g2=5 -> B12/B21 =', B12_over_B21(3, 5))
print('A21/B21 at 5.0e14 Hz =', A_over_B21(5.0e14), 'J s m^-3')
assert abs(tau - 10e-9) < 1e-12